In [2]:
import torch
import pandas as pd
import torch_geometric
from torch_geometric.data import Data, HeteroData
from torch_geometric.utils import to_undirected
from torch_geometric.transforms import RandomLinkSplit
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from sklearn.metrics import roc_auc_score, average_precision_score


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
df = pd.read_csv('../data/edges/clean_triples.csv')

In [22]:
def build_type_maps(df):
    maps = {}
    for t in pd.concat([df["type_entity_1"], df["type_entity_2"]]).unique():
        ids = pd.concat([
            df.loc[df["type_entity_1"] == t, "id_entity_1"],
            df.loc[df["type_entity_2"] == t, "id_entity_2"],
        ]).unique()
        maps[t] = {int(rid): i for i, rid in enumerate(ids)}
    return maps

def init_nodes(data, maps, emb_dim=64, device="cpu"):
    for node_type, m in maps.items():
        n = len(m)
        data[node_type].x = torch.randn(n, emb_dim, device=device)
    return data


def add_edges(data, df, maps, device="cpu", make_reverse=False):
    # сгруппируем по (type1, pred, type2)
    for (t1, pred, t2), g in df.groupby(["type_entity_1", "predicate", "type_entity_2"], sort=False):
        src = torch.tensor([maps[t1][int(x)] for x in g["id_entity_1"].tolist()],
                           dtype=torch.long, device=device)
        dst = torch.tensor([maps[t2][int(x)] for x in g["id_entity_2"].tolist()],
                           dtype=torch.long, device=device)
        ei = torch.stack([src, dst], dim=0)  # [2, E]
        data[(t1, pred, t2)].edge_index = ei

        if make_reverse:
            data[(t2, f"{pred}", t1)].edge_index = torch.stack([dst, src], dim=0)
    return data


def df_to_heterodata(df, emb_dim=64, device="cpu", make_reverse=False):
    df = df.copy()
    maps = build_type_maps(df)

    data = HeteroData()
    data = init_nodes(data, maps, emb_dim=emb_dim, device=device)
    data = add_edges(data, df, maps, device=device, make_reverse=make_reverse)

    return data, maps

data, maps = df_to_heterodata(df, emb_dim=64, make_reverse=True, device="cuda")
data

HeteroData(
  RNA={ x=[738, 64] },
  DNA={ x=[596, 64] },
  NucleicMixed={ x=[23, 64] },
  NucleicAmbigous={ x=[16, 64] },
  AA={ x=[70937, 64] },
  SmallMolecule={ x=[1001342, 64] },
  (RNA, interacts_with, AA)={ edge_index=[2, 340] },
  (AA, interacts_with, RNA)={ edge_index=[2, 340] },
  (DNA, interacts_with, AA)={ edge_index=[2, 323] },
  (AA, interacts_with, DNA)={ edge_index=[2, 323] },
  (NucleicMixed, interacts_with, AA)={ edge_index=[2, 10] },
  (AA, interacts_with, NucleicMixed)={ edge_index=[2, 10] },
  (NucleicAmbigous, interacts_with, AA)={ edge_index=[2, 2] },
  (AA, interacts_with, NucleicAmbigous)={ edge_index=[2, 2] },
  (DNA, interacts_with, SmallMolecule)={ edge_index=[2, 311] },
  (SmallMolecule, interacts_with, DNA)={ edge_index=[2, 311] },
  (RNA, interacts_with, SmallMolecule)={ edge_index=[2, 1461] },
  (SmallMolecule, interacts_with, RNA)={ edge_index=[2, 1461] },
  (NucleicAmbigous, interacts_with, SmallMolecule)={ edge_index=[2, 35] },
  (SmallMolecule, inter

In [8]:
def df_to_homo_two_rel(df: pd.DataFrame, emb_dim=64, device="cpu"):
    # 1) глобальные id для узлов: различаем по (entity_type, raw_id)
    #    (если ты хочешь 1 тип узлов, но НЕ сливать разные типы с одинаковым числом)
    nodes = pd.concat([
        df[["type_entity_1","id_entity_1"]].rename(columns={"type_entity_1":"t","id_entity_1":"id"}),
        df[["type_entity_2","id_entity_2"]].rename(columns={"type_entity_2":"t","id_entity_2":"id"}),
    ], axis=0).drop_duplicates()

    key = list(zip(nodes["t"].astype(str), nodes["id"].astype(int)))
    node_map = {k:i for i,k in enumerate(key)}
    num_nodes = len(node_map)

    # 2) edge_index
    src = [node_map[(str(t), int(i))] for t,i in zip(df["type_entity_1"], df["id_entity_1"])]
    dst = [node_map[(str(t), int(i))] for t,i in zip(df["type_entity_2"], df["id_entity_2"])]
    edge_index = torch.tensor([src, dst], dtype=torch.long)

    # 3) edge_type (2 предиката -> 0/1)
    preds = df["predicate"].astype(str).unique().tolist()
    pred2id = {p:i for i,p in enumerate(sorted(preds))}
    edge_type = torch.tensor([pred2id[p] for p in df["predicate"].astype(str)], dtype=torch.long)

    edge_index, edge_type = to_undirected(edge_index, edge_type)

    # 4) node features (рандом)
    x = torch.randn(num_nodes, emb_dim)
    data = Data(x=x, edge_index=edge_index)
    data.edge_type = edge_type  # [E]
    return data, node_map, pred2id




In [ ]:
data, node_map, pre2id = df_to_homo_two_rel(df, emb_dim=64)

interacts_mask = data.edge_type == 0
similar_mask = data.edge_type == 1

edge_index_interacts = data.edge_index[:, interacts_mask]
edge_index_similarity = data.edge_index[:, similar_mask]

#создаем временный граф
data_interacts = Data(
    x=data.x,
    edge_index=edge_index_interacts
)

#сплит ребер interacts
transform = RandomLinkSplit(
    num_val=0.1,
    num_test=0.1,
    is_undirected=True,
    add_negative_train_samples=True,
    neg_sampling_ratio = 1
)

train_data, val_data, test_data = transform(data_interacts)

#обратно добавляем has_similarity
for split_data in [train_data, val_data, test_data]:
    split_data.edge_index = torch.cat([split_data.edge_index, edge_index_similarity], dim=1)


In [5]:
class GraphSAGE(torch.nn.Module):

    def __init__(self, in_dim, hidden_dim):
        super().__init__()

        self.conv1 = SAGEConv(in_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x, edge_index):

        x = self.conv1(x, edge_index)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.conv2(x, edge_index)

        return x

def decode(z, edge_label_index):

    src = z[edge_label_index[0]]
    dst = z[edge_label_index[1]]

    return (src * dst).sum(dim=1)


def train():

    model.train()
    optimizer.zero_grad()

    z = model(train_data.x, train_data.edge_index)

    logits = decode(z, train_data.edge_label_index)
    loss_value = F.binary_cross_entropy_with_logits(logits, train_data.edge_label.float())

    loss_value.backward()
    optimizer.step()

    return loss_value.item()



@torch.no_grad()
def ranking_metrics(model, data, k_list=[1, 3, 10], N=1000):
    model.eval()
    z = model(data.x, data.edge_index)

    edge_label_index = data.edge_label_index[:,:N]

    num_edges = edge_label_index.size(1)

    ranks = []

    for i in range(num_edges):
        src = edge_label_index[0, i]
        dst = edge_label_index[1, i]

        # score всех возможных dst для данного src
        scores = (z[src] * z).sum(dim=1)  # [num_nodes]
        _, sorted_idx = torch.sort(scores, descending=True)

        # позиция правильного ребра
        rank = (sorted_idx == dst).nonzero(as_tuple=True)[0].item() + 1  # 1-based
        ranks.append(rank)

    ranks = torch.tensor(ranks, dtype=torch.float)

    mrr = torch.mean(1.0 / ranks).item()

    hits = {}
    for k in k_list:
        hits_k = torch.mean((ranks <= k).float()).item()
        hits[k] = hits_k

    return mrr, hits


In [10]:
train_data = train_data.to(device)
val_data = val_data.to(device)
test_data = test_data.to(device)

model = GraphSAGE(
    in_dim=train_data.x.size(1),
    hidden_dim=64
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(1, 201):

    loss_value = train()

    if epoch % 10 == 0:
        print(
            f"Epoch {epoch:03d} | "
            f"Loss {loss_value:.4f} | "
        )

Epoch 010 | Loss 0.7598 | 
Epoch 020 | Loss 0.6791 | 
Epoch 030 | Loss 0.6274 | 
Epoch 040 | Loss 0.5998 | 
Epoch 050 | Loss 0.5841 | 
Epoch 060 | Loss 0.5726 | 
Epoch 070 | Loss 0.5647 | 
Epoch 080 | Loss 0.5580 | 
Epoch 090 | Loss 0.5517 | 
Epoch 100 | Loss 0.5455 | 
Epoch 110 | Loss 0.5386 | 
Epoch 120 | Loss 0.5311 | 
Epoch 130 | Loss 0.5211 | 
Epoch 140 | Loss 0.5093 | 
Epoch 150 | Loss 0.4971 | 
Epoch 160 | Loss 0.4857 | 
Epoch 170 | Loss 0.4770 | 
Epoch 180 | Loss 0.4716 | 
Epoch 190 | Loss 0.4680 | 
Epoch 200 | Loss 0.4651 | 


In [11]:
mrr, hits = ranking_metrics(model, test_data, k_list=[1, 3, 10, 50])

print(f"MRR: {mrr:.4f}")
for k, v in hits.items():
    print(f"Hits@{k}: {v:.4f}")

MRR: 0.0054
Hits@1: 0.0000
Hits@3: 0.0040
Hits@10: 0.0150
Hits@50: 0.0360
